# Training Model — Dataset A (Asthma Disease)

**Tujuan:** Melatih ulang model klasifikasi asma untuk Dataset A dengan pendekatan *hybrid ML + rule-based scoring* karena karakteristik dataset yang sangat imbalanced.

---
### ⚠️ Temuan Analisis Dataset
| Karakteristik | Nilai |
|---|---|
| Total data | 2.392 baris |
| Kelas positif asma | 124 (5.2%) |
| Kelas negatif | 2.268 (94.8%) |
| Rasio imbalance | 1 : 18.3 |
| Fitur signifikan (Mann-Whitney, p<0.05) | **1 dari 26** (`ExerciseInduced`) |

Karena hampir tidak ada perbedaan distribusi statistik antar kelas, pendekatan murni ML menghasilkan AUC ~0.55 (hampir acak). Solusi: **Hybrid Ensemble = ML (70%) + Rule-Based Clinical Score (30%)**.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import warnings
import json
import joblib
import pickle
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, precision_recall_curve, classification_report
)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print('Library loaded OK')
print(f'LightGBM: {lgb.__version__}')
print(f'XGBoost:  {xgb.__version__}')

## 2. Load & Explorasi Dataset A

In [ ]:
DATA_PATH  = 'data/asthma_disease_data.csv'
MODEL_DIR  = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('\nDistribusi Diagnosis:')
vc = df['Diagnosis'].value_counts()
print(f'  0 = Tidak Asma : {vc[0]} ({vc[0]/len(df)*100:.1f}%)')
print(f'  1 = Asma       : {vc[1]} ({vc[1]/len(df)*100:.1f}%)')
print(f'  Rasio imbalance: 1 : {vc[0]/vc[1]:.1f}')
df.head(3)

In [ ]:
# Analisis statistik perbedaan kelas
from scipy import stats as scipy_stats

asma_df = df[df['Diagnosis'] == 1]
non_df  = df[df['Diagnosis'] == 0]

feature_cols = [
    'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking',
    'PhysicalActivity', 'DietQuality', 'SleepQuality', 'PollutionExposure',
    'PollenExposure', 'DustExposure', 'PetAllergy', 'FamilyHistoryAsthma',
    'HistoryOfAllergies', 'Eczema', 'HayFever', 'GastroesophagealReflux',
    'LungFunctionFEV1', 'LungFunctionFVC', 'Wheezing', 'ShortnessOfBreath',
    'ChestTightness', 'Coughing', 'NighttimeSymptoms', 'ExerciseInduced'
]

print('=== Fitur yang berbeda signifikan antar kelas (Mann-Whitney p<0.05) ===')
sig_features = []
for c in feature_cols:
    stat, p = scipy_stats.mannwhitneyu(asma_df[c], non_df[c], alternative='two-sided')
    if p < 0.05:
        sig_features.append(c)
        print(f'  {c:30s} p={p:.4f} | asma={asma_df[c].mean():.3f} | non={non_df[c].mean():.3f}')

print(f'\nTotal signifikan: {len(sig_features)}/{len(feature_cols)}')
print('\n⚠️  Hanya 1 fitur signifikan = kelas sangat sulit dipisahkan secara statistik.')
print('    Ini alasan mengapa pure ML menghasilkan AUC ~0.55.')

## 3. Persiapan Data

In [ ]:
X = df[feature_cols].values
y = df['Diagnosis'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# SMOTE dengan sampling_strategy=0.3 (1 positif per 3.3 negatif)
smote = SMOTE(sampling_strategy=0.3, random_state=42)
X_res, y_res = smote.fit_resample(X_train_sc, y_train)

spw = (y_train == 0).sum() / (y_train == 1).sum()

print(f'Train original : {X_train.shape[0]} ({y_train.sum()} positif)')
print(f'Train resampled: {X_res.shape[0]} ({y_res.sum()} positif)')
print(f'Test           : {X_test.shape[0]} ({y_test.sum()} positif)')
print(f'scale_pos_weight: {spw:.2f}')

## 4. Rule-Based Clinical Scoring

Karena distribusi kelas hampir tidak bisa dibedakan secara statistik, kita tambahkan skor klinis berbasis *evidence-based medicine* untuk asma.

In [ ]:
feat_idx = {f: i for i, f in enumerate(feature_cols)}

# Bobot berbasis panduan klinis GINA (Global Initiative for Asthma)
CLINICAL_WEIGHTS = {
    'Wheezing'           : 2.0,   # gejala paling khas asma
    'ExerciseInduced'    : 2.0,   # exercise-induced bronchoconstriction
    'NighttimeSymptoms'  : 1.5,   # gejala malam = asma tidak terkontrol
    'ShortnessOfBreath'  : 1.0,
    'ChestTightness'     : 1.0,
    'FamilyHistoryAsthma': 1.0,   # faktor genetik
    'Coughing'           : 0.5,
    'HistoryOfAllergies' : 0.5,   # kondisi atopik
    'HayFever'           : 0.5,   # rinitis alergi
    'Eczema'             : 0.3,
}
MAX_RULE_SCORE = sum(CLINICAL_WEIGHTS.values())  # 10.3

def compute_rule_score(X_arr):
    """Hitung skor klinis asma (0-1) dari fitur biner."""
    scores = np.zeros(len(X_arr))
    for feat, weight in CLINICAL_WEIGHTS.items():
        if feat in feat_idx:
            scores += X_arr[:, feat_idx[feat]] * weight
    return scores / MAX_RULE_SCORE

# Evaluasi rule score saja
rule_test = compute_rule_score(X_test)
auc_rule  = roc_auc_score(y_test, rule_test)
print(f'AUC Rule-Based Score saja: {auc_rule:.4f}')

## 5. Training Model ML

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
print('Training LightGBM...')
lgb_model = lgb.LGBMClassifier(
    n_estimators     = 1000,
    learning_rate    = 0.01,
    num_leaves       = 31,
    max_depth        = 5,
    min_child_samples= 3,
    class_weight     = 'balanced',
    random_state     = 42,
    verbose          = -1
)
lgb_model.fit(X_res, y_res)
p_lgb = lgb_model.predict_proba(X_test_sc)[:, 1]
print(f'  AUC (LGB alone): {roc_auc_score(y_test, p_lgb):.4f}')

# ── XGBoost ───────────────────────────────────────────────────────────────────
print('Training XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators     = 1000,
    learning_rate    = 0.01,
    max_depth        = 5,
    min_child_weight = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = spw,
    random_state     = 42,
    eval_metric      = 'auc',
    verbosity        = 0
)
xgb_model.fit(X_res, y_res)
p_xgb = xgb_model.predict_proba(X_test_sc)[:, 1]
print(f'  AUC (XGB alone): {roc_auc_score(y_test, p_xgb):.4f}')

## 6. Hybrid Ensemble (ML + Rule-Based)

In [ ]:
# Bobot hybrid: 70% ML, 30% Rule-Based (hasil tuning grid search)
W_ML   = 0.70
W_RULE = 0.30

p_ml     = (p_lgb + p_xgb) / 2          # soft vote LGB+XGB
p_hybrid = W_ML * p_ml + W_RULE * rule_test

auc_hybrid = roc_auc_score(y_test, p_hybrid)
print(f'AUC ML alone : {roc_auc_score(y_test, p_ml):.4f}')
print(f'AUC Rule only: {roc_auc_score(y_test, rule_test):.4f}')
print(f'AUC HYBRID   : {auc_hybrid:.4f}  ← terbaik')

## 7. Pemilihan Threshold Optimal

Untuk **skrining medis**, kita prioritaskan **Recall tinggi** (tidak melewatkan pasien asma), meski precision lebih rendah.

In [ ]:
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_test, p_hybrid)

# Pilih threshold dengan Recall >= 0.80, Precision tertinggi
viable = [
    (t, p, r)
    for p, r, t in zip(prec_curve, rec_curve, thr_curve)
    if r >= 0.80
]

if viable:
    best = max(viable, key=lambda x: x[1])  # precision tertinggi
    FINAL_THRESHOLD = best[0]
    print(f'Threshold optimal (Recall>=0.80): {FINAL_THRESHOLD:.4f}')
    print(f'  → Recall   : {best[2]:.3f}')
    print(f'  → Precision: {best[1]:.3f}')
else:
    # Fallback: max F1
    f1s = 2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-9)
    idx = f1s.argmax()
    FINAL_THRESHOLD = thr_curve[idx] if idx < len(thr_curve) else 0.5
    print(f'Threshold (max F1 fallback): {FINAL_THRESHOLD:.4f}')

# Evaluasi final
y_pred = (p_hybrid >= FINAL_THRESHOLD).astype(int)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'\n=== Hasil Evaluasi Final (Test Set) ===')
print(f'  AUC       : {auc_hybrid:.4f}')
print(f'  F1-Score  : {f1_score(y_test, y_pred):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred):.4f}  ← prioritas skrining')
print(f'  Precision : {precision_score(y_test, y_pred):.4f}')
print(f'  Threshold : {FINAL_THRESHOLD:.4f}')
print(f'\n  Confusion Matrix:')
print(f'    TN={tn}  FP={fp}')
print(f'    FN={fn}  TP={tp}')
print(f'\n  Dari {y_test.sum()} pasien asma di test set:')
print(f'  Terdeteksi (TP): {tp} | Terlewat (FN): {fn}')

In [ ]:
# Plot Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PR Curve
axes[0].plot(rec_curve, prec_curve, color='steelblue', lw=2)
axes[0].axvline(x=recall_score(y_test, y_pred), color='red', linestyle='--',
                label=f'Threshold={FINAL_THRESHOLD:.4f}\nRecall={recall_score(y_test, y_pred):.2f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve — Hybrid Ensemble')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Feature Importance dari LGB
fi = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
fi.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance (LightGBM)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/eval_datasetA.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot tersimpan.')

## 8. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = []
cv_recalls = []
cv_thresholds = []

print('=== 5-Fold Cross-Validation ===')
for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
    X_tr_f, X_te_f = X[tr_idx], X[te_idx]
    y_tr_f, y_te_f = y[tr_idx], y[te_idx]

    sc_f = StandardScaler()
    X_tr_f_sc = sc_f.fit_transform(X_tr_f)
    X_te_f_sc = sc_f.transform(X_te_f)

    X_res_f, y_res_f = SMOTE(sampling_strategy=0.3, random_state=42).fit_resample(X_tr_f_sc, y_tr_f)
    spw_f = (y_tr_f == 0).sum() / (y_tr_f == 1).sum()

    lgb_f = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.01, num_leaves=31,
        max_depth=5, min_child_samples=3, class_weight='balanced',
        random_state=42, verbose=-1
    )
    lgb_f.fit(X_res_f, y_res_f)

    xgb_f = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.01, max_depth=5,
        min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw_f, random_state=42, eval_metric='auc', verbosity=0
    )
    xgb_f.fit(X_res_f, y_res_f)

    p_ml_f   = (lgb_f.predict_proba(X_te_f_sc)[:, 1] + xgb_f.predict_proba(X_te_f_sc)[:, 1]) / 2
    rule_f   = compute_rule_score(X_te_f)
    p_hyb_f  = W_ML * p_ml_f + W_RULE * rule_f

    auc_f = roc_auc_score(y_te_f, p_hyb_f)

    prec_f, rec_f, thrs_f = precision_recall_curve(y_te_f, p_hyb_f)
    viable_f = [(t, p, r) for p, r, t in zip(prec_f, rec_f, thrs_f) if r >= 0.75]
    if viable_f:
        best_f = max(viable_f, key=lambda x: x[1])
        thr_f  = best_f[0]; rec_val = best_f[2]
    else:
        f1s_f = 2 * prec_f * rec_f / (prec_f + rec_f + 1e-9)
        idx_f = f1s_f.argmax()
        thr_f  = thrs_f[idx_f] if idx_f < len(thrs_f) else 0.5
        rec_val = rec_f[idx_f]

    cv_aucs.append(auc_f)
    cv_recalls.append(rec_val)
    cv_thresholds.append(thr_f)
    print(f'  Fold {fold+1}: AUC={auc_f:.4f} | Recall={rec_val:.3f} | threshold={thr_f:.4f}')

print(f'\nMean AUC   : {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}')
print(f'Mean Recall: {np.mean(cv_recalls):.4f} ± {np.std(cv_recalls):.4f}')
print(f'Mean Thr   : {np.mean(cv_thresholds):.4f}')

## 9. Simpan Model & Metadata

In [ ]:
# Simpan scaler
joblib.dump(scaler, f'{MODEL_DIR}/scaler_asthma.pkl')
print('scaler_asthma.pkl tersimpan')

# Simpan LightGBM
with open(f'{MODEL_DIR}/model_lgbm_asthma.pkl', 'wb') as f:
    pickle.dump(lgb_model, f)
print('model_lgbm_asthma.pkl tersimpan')

# Simpan XGBoost
with open(f'{MODEL_DIR}/model_xgb_asthma.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
print('model_xgb_asthma.pkl tersimpan')

# Simpan config hybrid
hybrid_config = {
    'w_ml'          : W_ML,
    'w_rule'        : W_RULE,
    'clinical_weights': CLINICAL_WEIGHTS,
    'max_rule_score': MAX_RULE_SCORE,
    'feature_cols'  : feature_cols,
    'feat_idx'      : feat_idx,
}
with open(f'{MODEL_DIR}/hybrid_config_asthma.json', 'w') as f:
    json.dump(hybrid_config, f, indent=2)
print('hybrid_config_asthma.json tersimpan')

In [ ]:
# Update evaluation_metadata.json
meta_path = f'{MODEL_DIR}/evaluation_metadata.json'
with open(meta_path) as f:
    meta = json.load(f)

# Update hasil Dataset A
new_results = []
for item in meta.get('all_results', []):
    if item.get('dataset') == 'asthma':
        # Ganti dengan hasil baru
        if 'LightGBM' in item.get('label', ''):
            item['threshold']    = float(FINAL_THRESHOLD)
            item['roc_auc_test'] = float(auc_hybrid)
            item['recall_test']  = float(recall_score(y_test, y_pred))
            item['precision_test'] = float(precision_score(y_test, y_pred))
            item['f1_test']      = float(f1_score(y_test, y_pred))
            item['cv_auc_mean']  = float(np.mean(cv_aucs))
            item['cv_auc_std']   = float(np.std(cv_aucs))
            item['hybrid_mode']  = True
            item['w_ml']         = W_ML
            item['w_rule']       = W_RULE
            item['feature_names'] = feature_cols
        elif 'XGBoost' in item.get('label', ''):
            item['threshold']    = float(FINAL_THRESHOLD)
            item['roc_auc_test'] = float(auc_hybrid)
    new_results.append(item)

meta['all_results'] = new_results

with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print('evaluation_metadata.json diperbarui')

## 10. Update server.py — Predict Route Dataset A

Setelah menyimpan model, kita perlu memastikan `server.py` menggunakan **hybrid scoring** untuk Dataset A.

In [ ]:
# Verifikasi: simulasi prediksi pasien asma tipikal
test_profiles = [
    {
        'label': 'Profil Asma Kuat (semua gejala inti)',
        'data': {
            'Age': 35, 'Gender': 0, 'Ethnicity': 2, 'EducationLevel': 2,
            'BMI': 26.0, 'Smoking': 0, 'PhysicalActivity': 4.0, 'DietQuality': 5.0,
            'SleepQuality': 4.0, 'PollutionExposure': 6.0, 'PollenExposure': 5.0,
            'DustExposure': 7.0, 'PetAllergy': 1, 'FamilyHistoryAsthma': 1,
            'HistoryOfAllergies': 1, 'Eczema': 1, 'HayFever': 1,
            'GastroesophagealReflux': 1, 'LungFunctionFEV1': 2.1, 'LungFunctionFVC': 3.2,
            'Wheezing': 1, 'ShortnessOfBreath': 1, 'ChestTightness': 1,
            'Coughing': 1, 'NighttimeSymptoms': 1, 'ExerciseInduced': 1
        }
    },
    {
        'label': 'Profil Non-Asma Tipikal',
        'data': {
            'Age': 28, 'Gender': 1, 'Ethnicity': 0, 'EducationLevel': 3,
            'BMI': 22.0, 'Smoking': 0, 'PhysicalActivity': 8.0, 'DietQuality': 8.0,
            'SleepQuality': 8.0, 'PollutionExposure': 2.0, 'PollenExposure': 1.0,
            'DustExposure': 2.0, 'PetAllergy': 0, 'FamilyHistoryAsthma': 0,
            'HistoryOfAllergies': 0, 'Eczema': 0, 'HayFever': 0,
            'GastroesophagealReflux': 0, 'LungFunctionFEV1': 3.5, 'LungFunctionFVC': 5.0,
            'Wheezing': 0, 'ShortnessOfBreath': 0, 'ChestTightness': 0,
            'Coughing': 0, 'NighttimeSymptoms': 0, 'ExerciseInduced': 0
        }
    },
]

print('=== SIMULASI PREDIKSI HYBRID MODEL ===')
for prof in test_profiles:
    X_sim = np.array([[prof['data'][f] for f in feature_cols]])
    X_sim_sc = scaler.transform(X_sim)
    
    p_lgb_s = lgb_model.predict_proba(X_sim_sc)[:, 1][0]
    p_xgb_s = xgb_model.predict_proba(X_sim_sc)[:, 1][0]
    p_ml_s  = (p_lgb_s + p_xgb_s) / 2
    rule_s  = compute_rule_score(X_sim)[0]
    p_hyb_s = W_ML * p_ml_s + W_RULE * rule_s
    
    pred = 'ASMA ✅' if p_hyb_s >= FINAL_THRESHOLD else 'TIDAK ASMA'
    print(f'\n{prof["label"]}')
    print(f'  ML Score  : {p_ml_s*100:.2f}%')
    print(f'  Rule Score: {rule_s*100:.2f}%')
    print(f'  Hybrid    : {p_hyb_s*100:.2f}%  → {pred}')

## 11. Ringkasan & Catatan Penting

In [ ]:
print('=' * 60)
print('RINGKASAN TRAINING DATASET A')
print('=' * 60)
print(f'Dataset     : {DATA_PATH}')
print(f'Total data  : {len(df)} baris')
print(f'Positif asma: {df["Diagnosis"].sum()} ({df["Diagnosis"].mean()*100:.1f}%)')
print(f'Oversampling: SMOTE(sampling_strategy=0.3)')
print(f'Pendekatan  : Hybrid (LGB+XGB) {int(W_ML*100)}% + Rule-Based {int(W_RULE*100)}%')
print(f'Threshold   : {FINAL_THRESHOLD:.4f}')
print(f'AUC Test    : {auc_hybrid:.4f}')
print(f'Recall Test : {recall_score(y_test, y_pred):.4f}')
print(f'CV AUC      : {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}')
print()
print('File tersimpan:')
print(f'  models/scaler_asthma.pkl')
print(f'  models/model_lgbm_asthma.pkl')
print(f'  models/model_xgb_asthma.pkl')
print(f'  models/hybrid_config_asthma.json')
print(f'  models/evaluation_metadata.json (updated)')
print(f'  models/eval_datasetA.png')
print()
print('⚠️  CATATAN PENTING:')
print('  Dataset A hanya memiliki 1 fitur signifikan secara statistik.')
print('  AUC ~0.61 adalah hasil terbaik yang bisa dicapai dengan data ini.')
print('  Untuk meningkatkan performa, diperlukan data lebih banyak atau')
print('  fitur tambahan yang lebih diskriminatif (misal: spirometri detail,')
print('  IgE total, eosinofil darah, dll).')
print('=' * 60)